# Tarea M53 – Ali Vega

# Comandos avanzados en Spark con Python

Práctica de Big Data: partición, exportación y conexión con PySpark

## Introducción

En esta práctica se utilizan comandos avanzados de Spark mediante PySpark, trabajando desde un notebook de Python. El ejercicio se enfoca en importar datos desde Spark, establecer una conexión entre Python y el servicio Spark, y ejecutar comandos que permitan consultar, procesar y visualizar información.

También se aplican operaciones relacionadas con el procesamiento distribuido, como la partición de datos y la exportación de resultados. El objetivo es comprender cómo Python puede conectarse con Spark para trabajar con información dentro de un entorno de Big Data.

## 1. Importación de librerías

En esta sección se importan las librerías necesarias para trabajar con Spark desde Python.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, avg

## 2. Conexión de Python con Spark

Se crea una sesión de Spark utilizando `SparkSession`. Esta sesión permite conectar Python con el servicio de Spark para procesar datos.

In [2]:
spark = SparkSession.builder \
    .appName("PracticaBigDataSpark") \
    .getOrCreate()

spark

## 3. Importación de datos desde Spark

Se importa el archivo del dataset Housing en formato CSV. Para esta lectura se indica que el archivo contiene encabezados y que Spark debe inferir automáticamente el tipo de dato de cada columna.

In [3]:
housing = spark.read.csv(
    r"C:\Users\alive\Downloads\kc_house_data (1).csv",
    header=True,
    inferSchema=True
)

## 4. Visualización inicial de los datos

Se muestran las primeras filas del DataFrame para revisar que la importación se realizó correctamente.

In [4]:
housing.show(5)

+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date|   price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|7129300520|20141013T000000|221900.0|       3|      1.0|       1180|    5650|   1.0|         0|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|
|6414100192|20141209T000000|538000.0|       3|     2.25|       2570|    7242|   2.0|         0|   0|        3|    7|      2170|          400|   

## 5. Revisión de estructura del DataFrame

Se utiliza `printSchema()` para conocer las columnas del dataset y el tipo de dato asignado por Spark.

In [5]:
housing.printSchema()

root
 |-- id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft_living: integer (nullable = true)
 |-- sqft_lot: integer (nullable = true)
 |-- floors: double (nullable = true)
 |-- waterfront: integer (nullable = true)
 |-- view: integer (nullable = true)
 |-- condition: integer (nullable = true)
 |-- grade: integer (nullable = true)
 |-- sqft_above: integer (nullable = true)
 |-- sqft_basement: integer (nullable = true)
 |-- yr_built: integer (nullable = true)
 |-- yr_renovated: integer (nullable = true)
 |-- zipcode: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- sqft_living15: integer (nullable = true)
 |-- sqft_lot15: integer (nullable = true)



## 6. Selección de columnas principales

Se seleccionan algunas columnas importantes del dataset para visualizar información relacionada con precio, habitaciones, baños, tamaño y código postal.

In [6]:
housing.select(
    "price",
    "bedrooms",
    "bathrooms",
    "sqft_living",
    "zipcode"
).show(10)

+---------+--------+---------+-----------+-------+
|    price|bedrooms|bathrooms|sqft_living|zipcode|
+---------+--------+---------+-----------+-------+
| 221900.0|       3|      1.0|       1180|  98178|
| 538000.0|       3|     2.25|       2570|  98125|
| 180000.0|       2|      1.0|        770|  98028|
| 604000.0|       4|      3.0|       1960|  98136|
| 510000.0|       3|      2.0|       1680|  98074|
|1225000.0|       4|      4.5|       5420|  98053|
| 257500.0|       3|     2.25|       1715|  98003|
| 291850.0|       3|      1.5|       1060|  98198|
| 229500.0|       3|      1.0|       1780|  98146|
| 323000.0|       3|      2.5|       1890|  98038|
+---------+--------+---------+-----------+-------+
only showing top 10 rows


## 7. Ordenamiento de datos

Se ordena el DataFrame de acuerdo con la columna `zipcode`, para observar los registros organizados por código postal.

In [7]:
housing.orderBy("zipcode").show(20)

+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date|   price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+--------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|3353400860|20140717T000000|249900.0|       3|     1.75|       2080|   12522|   1.0|         0|   0|        5|    6|      2080|            0|    1950|           0|  98001| 47.267| -122.25|         1690|     11200|
|3354400060|20150501T000000|238000.0|       2|      1.0|       1088|    8453|   1.0|         0|   0|        3|    6|      1088|            0|   

## 8. Agrupamiento por código postal

Se agrupan los datos por `zipcode` para identificar cuántas casas existen en cada código postal.

In [8]:
housing.groupBy("zipcode") \
    .agg(count("*").alias("total_casas")) \
    .orderBy("total_casas", ascending=False) \
    .show(20)

+-------+-----------+
|zipcode|total_casas|
+-------+-----------+
|  98103|        602|
|  98038|        590|
|  98115|        583|
|  98052|        574|
|  98117|        553|
|  98042|        548|
|  98034|        545|
|  98118|        508|
|  98023|        499|
|  98006|        498|
|  98133|        494|
|  98059|        468|
|  98058|        455|
|  98155|        446|
|  98074|        441|
|  98033|        432|
|  98027|        412|
|  98125|        410|
|  98056|        406|
|  98053|        405|
+-------+-----------+
only showing top 20 rows


## 9. Promedio de precio y tamaño por código postal

Se calcula el precio promedio y el tamaño promedio de las viviendas agrupadas por código postal.

In [9]:
housing.groupBy("zipcode") \
    .agg(
        avg("price").alias("promedio_precio"),
        avg("sqft_living").alias("promedio_tamano")
    ) \
    .orderBy("zipcode") \
    .show(20)

+-------+------------------+------------------+
|zipcode|   promedio_precio|   promedio_tamano|
+-------+------------------+------------------+
|  98001| 280804.6906077348|1900.8563535911603|
|  98002| 234284.0351758794|1627.7437185929648|
|  98003|294111.27857142856|1928.8821428571428|
|  98004|1355927.0820189274|2909.0220820189274|
|  98005|        810164.875|2656.8035714285716|
|  98006| 859684.7791164658|2888.2951807228915|
|  98007|  617105.085106383|2182.0567375886526|
|  98008| 645507.3780918728|2133.4452296819786|
|  98010|         423665.99|           2137.59|
|  98011| 490351.4666666667| 2253.097435897436|
|  98014| 455617.1129032258| 2117.967741935484|
|  98019|424788.74736842106| 2171.557894736842|
|  98022|315709.30341880344|1832.3247863247864|
|  98023|286732.79158316634|1989.7294589178357|
|  98024| 580526.7901234567|2337.4691358024693|
|  98027| 616990.5922330098| 2514.609223300971|
|  98028|  462480.035335689|2122.7243816254418|
|  98029| 612653.6105919003|2284.2056074

## 10. Agrupamiento por habitaciones y baños

Se agrupan los datos por número de habitaciones y baños para calcular el precio promedio de las viviendas.

In [10]:
housing.groupBy(
    "bedrooms",
    "bathrooms"
).agg(
    avg("price").alias("precio_promedio")
).orderBy(
    "bedrooms",
    "bathrooms"
).show(20)

+--------+---------+------------------+
|bedrooms|bathrooms|   precio_promedio|
+--------+---------+------------------+
|       0|      0.0| 520371.4285714286|
|       0|     0.75|          265000.0|
|       0|      1.0|          228000.0|
|       0|      1.5|          288000.0|
|       0|      2.5| 299983.3333333333|
|       1|      0.0| 279666.6666666667|
|       1|      0.5|          255000.0|
|       1|     0.75| 251053.7037037037|
|       1|      1.0|316629.95652173914|
|       1|     1.25|          881750.0|
|       1|      1.5| 325745.8333333333|
|       1|     1.75|          366050.0|
|       1|      2.0| 297166.6666666667|
|       1|     2.25|          436225.0|
|       1|      2.5|          489500.0|
|       2|      0.5|          191000.0|
|       2|     0.75|          327513.5|
|       2|      1.0|348072.80102695763|
|       2|     1.25| 468983.3333333333|
|       2|      1.5| 410121.4387755102|
+--------+---------+------------------+
only showing top 20 rows


## 11. Partición de datos en Spark

Se utiliza `repartition()` para dividir el DataFrame en 4 particiones. Esto permite simular el procesamiento distribuido de datos dentro de Spark.

In [11]:
housing_partition = housing.repartition(4)

## 12. Verificación del número de particiones

Se verifica el número de particiones del DataFrame utilizando `getNumPartitions()`.

In [12]:
housing_partition.rdd.getNumPartitions()

4

## 13. Exportación de información

Se exporta una muestra de los datos procesados. Debido a que en el entorno local la escritura directa desde Spark puede presentar errores de permisos o configuración, se convierte una muestra del DataFrame a Pandas y posteriormente se guarda en formato CSV desde Python.

In [13]:
housing_partition.limit(100).toPandas().to_csv(
    "output_housing_python.csv",
    index=False
)

In [14]:
print("Exportación realizada correctamente desde Python")

Exportación realizada correctamente desde Python


## 14. Lectura del archivo exportado

Se lee nuevamente el archivo CSV generado para comprobar que la exportación se realizó correctamente.

In [15]:
import pandas as pd

archivo_exportado = pd.read_csv("output_housing_python.csv")
archivo_exportado.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,5466700450,20141015T000000,250000.0,4,1.75,1860,7350,1.0,0,0,...,7,1090,770,1977,0,98031,47.3979,-122.174,1710,7350
1,8856004327,20140509T000000,248000.0,4,3.00,2163,5883,2.0,0,0,...,7,2163,0,2006,0,98001,47.2734,-122.251,1700,10143
2,1005000062,20140801T000000,299000.0,2,1.00,1040,4600,1.0,0,0,...,6,1040,0,1950,0,98118,47.5387,-122.277,1390,5897
3,705700240,20150218T000000,380000.0,4,2.50,2320,10079,2.0,0,0,...,7,2320,0,1994,0,98038,47.3828,-122.026,2010,7438
4,7349650330,20140609T000000,270000.0,4,2.75,1990,7252,1.0,0,0,...,7,1270,720,1999,0,98002,47.2839,-122.202,2100,7535


## 15. Ejemplo de consulta con Python y Spark

Se ejecuta una consulta sencilla desde Python utilizando Spark para obtener información específica del DataFrame.

In [16]:
housing.select(
    "zipcode",
    "price",
    "bedrooms",
    "bathrooms"
).where(
    housing.price > 500000
).show(20)

+-------+---------+--------+---------+
|zipcode|    price|bedrooms|bathrooms|
+-------+---------+--------+---------+
|  98125| 538000.0|       3|     2.25|
|  98136| 604000.0|       4|      3.0|
|  98074| 510000.0|       3|      2.0|
|  98053|1225000.0|       4|      4.5|
|  98007| 662500.0|       3|      2.5|
|  98107| 530000.0|       5|      2.0|
|  98126| 650000.0|       4|      3.0|
|  98040|2000000.0|       3|     2.75|
|  98119| 937000.0|       3|     1.75|
|  98112| 667000.0|       3|      1.0|
|  98052| 719000.0|       4|      2.5|
|  98027| 580500.0|       3|      2.5|
|  98117| 687500.0|       4|     1.75|
|  98117| 535000.0|       3|      1.0|
|  98115| 696000.0|       3|      2.5|
|  98052| 550000.0|       4|      1.0|
|  98107| 640000.0|       4|      2.0|
|  98056| 605000.0|       4|      2.5|
|  98074| 625000.0|       4|      2.5|
|  98166| 775000.0|       4|     2.25|
+-------+---------+--------+---------+
only showing top 20 rows


## 16. Conclusión

En esta práctica se trabajó con PySpark para conectar Python con Spark y realizar operaciones básicas y avanzadas de procesamiento de datos. Se importó información desde un archivo CSV, se consultaron registros, se realizaron agrupamientos, ordenamientos y cálculos de promedios.

Además, se aplicó la partición del DataFrame para representar el procesamiento distribuido y se realizó la exportación de una muestra de los datos procesados. Con esto se refuerza el uso de Spark como herramienta para trabajar con información en entornos de Big Data.